In [10]:
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col, trim, lower, regexp_replace
import os

In [11]:
#exemple of path_data
path_landing_2 = "../../../data/interim/Kaggle/___________datasets_Kaggle_csv_twitter_csv" 
path_landing = "../../../data/interim/delta_lake" 

In [12]:
mongo_connector_jar = "/home/provira/Documents/TFM/TFM/notebooks/P2/trusted_zone/jars/mongo-spark-connector_2.12-3.0.1.jar"
mongo_driver_jar = "/home/provira/Documents/TFM/TFM/notebooks/P2/trusted_zone/jars/mongo-java-driver-3.12.10.jar"

In [13]:
builder = SparkSession.builder \
    .appName("Trusted_Zone") \
    .config("spark.jars", f"{mongo_connector_jar},{mongo_driver_jar}") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017") \
    .config("spark.mongodb.write.connection.uri", "mongodb://localhost:27017")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [14]:
spark.sparkContext._jsc.sc().listJars()

JavaObject id=o69

In [15]:
def readFromDeltaLake(path_landing):
    print(path_landing)
    return spark.read.format("delta").load(path_landing)

# Preprocessing

In [16]:
from pyspark.sql.types import StructType, StringType
from pyspark.sql.functions import col, from_json, decode

def cleanCSV(df_csv):
    print(df_csv.columns)
    print(df_csv.head())

    # 1️⃣ Define the schema of your JSON payload
    schema = StructType() \
        .add("source", StringType()) \
        .add("author", StringType()) \
        .add("message", StringType()) \
        .add("published_at", StringType())

    
    df_csv_clean = df_csv.dropna()


    # 2️⃣ Decode the binary column and parse JSON
    df_parsed = (
        df_csv_clean
        .withColumn("value_str", decode(col("value"), "utf-8"))        # convert bytes → string
        .withColumn("json", from_json(col("value_str"), schema))       # parse string → JSON struct
    )

    print(df_parsed.head())

    df_csv_clean = df_csv.dropna()

    rename_map_2 = {
        "_c0": "id",
        "Emotion": "emotion",
        "sentiment": "emotion",
        "tweet": "text",
        "label": "emotion",
    }

    rename_map = {
        "key": "id",
        "value": "emotion",
    }

    for old, new in rename_map.items():
        if old in df_csv_clean.columns:
            df_csv_clean = df_csv_clean.withColumnRenamed(old, new)
    print(df_csv_clean.head())
    # Only apply text processing if 'text' exists
    if "text" in df_csv_clean.columns:
        df_csv_clean = df_csv_clean \
            .withColumn("text", trim(col("text"))) \
            .withColumn("text", lower(col("text"))) \
            .withColumn("text", regexp_replace(col("text"), r"\bi m\b", "i'm")) \
            .withColumn("text", regexp_replace(col("text"), r"[^a-zA-Z0-9\s']", ""))  # keep letters, digits, spaces, apostrophes
    return df_csv_clean

In [17]:
#print(df_csv_clean.head())


In [18]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

#nltk.download('punkt')
#nltk.download('stopwords')
def tokenizer(df_csv_clean):
    stop_words = set(stopwords.words('english'))

    # Tokenizar
    tokenizer = Tokenizer(inputCol="text", outputCol="words")
    df_words = tokenizer.transform(df_csv_clean) #.limit(1000) limited to 1000 rows for performance

    # Eliminar stopwords (solo en inglés por defecto, pero puedes pasar las tuyas)
    remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
    remover.setStopWords(list(stop_words))
    df_filtered = remover.transform(df_words)

    #Stemming
    stemmer = SnowballStemmer("english")



    def stem_tokens(tokens):
        return [stemmer.stem(token) for token in tokens]
    
    stem_udf = udf(stem_tokens, ArrayType(StringType()))

    df_stemmed = df_filtered.withColumn("stemmed_words", stem_udf("filtered_words"))

    #df_stemmed.select("text", "filtered_words", "stemmed_words").show(truncate=False)

    return df_stemmed
    



In [19]:
from pyspark.ml.feature import CountVectorizer, IDF

def tf_idf(df_stemmed):

    # Paso 1: Crear el CountVectorizer para extraer el vocabulario y conteo de tokens
    cv = CountVectorizer(inputCol="stemmed_words", outputCol="raw_features")
    cv_model = cv.fit(df_stemmed)             # Entrenas el modelo con el vocabulario
    df_featurized = cv_model.transform(df_stemmed)  # Transformas el DataFrame

    # Paso 2: Calcular TF-IDF a partir del conteo
    idf = IDF(inputCol="raw_features", outputCol="tfidf_features")
    idf_model = idf.fit(df_featurized)          # Ajustar IDF sobre los datos
    df_tfidf = idf_model.transform(df_featurized) # Transformar con TF-IDF

    # Mostrar resultados
    #df_tfidf.select("stemmed_words", "raw_features", "tfidf_features").show(truncate=False)
    df_tfidf.printSchema()

    return df_tfidf


# Store in MongoDB

In [20]:

from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, FloatType
from pyspark.ml.linalg import VectorUDT

def df_clean(df_tfidf):
    def vector_to_array(v):
        return v.toArray().tolist() if v else None

    vector_to_array_udf = udf(vector_to_array, ArrayType(FloatType()))

    df_tfidf_safe = df_tfidf \
        .withColumn("raw_features_array", vector_to_array_udf("raw_features")) \
        .withColumn("tfidf_features_array", vector_to_array_udf("tfidf_features"))
    return df_tfidf_safe


In [21]:
def storeInMongoDB(df_tfidf_safe):
    df_tfidf_safe.select(
        "text", "Emotion", "words", "filtered_words", "stemmed_words",
        "raw_features_array", "tfidf_features_array"
    ).write \
        .format("mongo") \
        .option("uri", "mongodb://localhost:27017") \
        .option("database", "tfm-trusted-zone") \
        .option("collection", "tf-idf") \
        .mode("append") \
        .save()

# Pipeline

In [22]:
cleanCSV_udf = udf(cleanCSV, ArrayType(StringType()))
tokenizer_udf = udf(tokenizer, ArrayType(StringType()))
df_clean_udf = udf(df_clean, ArrayType(StringType()))  # Update return type if it's different

In [23]:
types = ['csv', 'parquet', 'json', 'txt']
sources = ['Kaggle', 'uci', 'AWS']

In [24]:
path_data = "./../../../data/interim/"

for source in sources:
    path = os.path.join(path_data, source)

    if os.path.isdir(path):
        for folder in os.listdir(path):
            full_path = os.path.join(path, folder)
            #print(f"Processing folder: {full_path}/")

            try:
                full_path = path_landing
                print(f"Processing folder: {full_path}/")
                df_csv = readFromDeltaLake(full_path)

                df_csv_clean = cleanCSV(df_csv) #  .limit(10) Limit to 10 rows for performance
                df_stemmed = tokenizer(df_csv_clean)
                df_tfidf = tf_idf(df_stemmed)
                df_tfidf_safe = df_clean(df_tfidf)

                storeInMongoDB(df_tfidf_safe)

            except Exception as e:
                print(f"Error processing folder {full_path}: {e}")

Processing folder: ../../../data/interim/delta_lake/
../../../data/interim/delta_lake


KeyboardInterrupt: 